In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [19]:
df_full = pd.read_csv('cocktail_dataset.csv')

print(f"📊 Dataset loaded: {len(df_full)} total recipes")
print(f"📝 Columns in dataset: {len(df_full.columns)}")

# Define flavor categories (target variables)
flavor_categories = ['bittersweet', 'citrus', 'creamy', 'floral', 'fruity', 'herbal', 'savoury', 'spicy', 'sweet']

print(f"\n🎯 Target flavor categories: {flavor_categories}")

has_flavor_tags = df_full[flavor_categories].sum(axis=1) > 0
df = df_full[has_flavor_tags].copy()

# Identify ingredient columns (all columns ending with '_pct')
ingredient_columns = [col for col in df.columns if col.endswith('_pct')]
print(f"\n🍹 Ingredient columns found: {len(ingredient_columns)}")

# Verify all flavor categories exist in the dataframe
missing_flavors = [f for f in flavor_categories if f not in df.columns]
if missing_flavors:
    print(f"❌ ERROR: Missing flavor columns: {missing_flavors}")
    print("Please check your column names.")
else:
    print("✅ All flavor categories found in dataset")

# Check for missing values
print(f"\n🔍 Checking data quality:")
print(f"   Total recipes: {len(df)}")
print(f"   Recipes with ingredient data: {df[ingredient_columns].notna().all(axis=1).sum()}")
print(f"   Recipes with flavor tags: {df[flavor_categories].notna().all(axis=1).sum()}")

# Separate features (ingredients) and targets (flavors)
X = df[ingredient_columns].copy()
y = df[flavor_categories].copy()

print(f"\n📐 Data shapes:")
print(f"   Features (X): {X.shape}")
print(f"   Targets (y): {y.shape}")

📊 Dataset loaded: 4602 total recipes
📝 Columns in dataset: 256

🎯 Target flavor categories: ['bittersweet', 'citrus', 'creamy', 'floral', 'fruity', 'herbal', 'savoury', 'spicy', 'sweet']

🍹 Ingredient columns found: 242
✅ All flavor categories found in dataset

🔍 Checking data quality:
   Total recipes: 2692
   Recipes with ingredient data: 2692
   Recipes with flavor tags: 2692

📐 Data shapes:
   Features (X): (2692, 242)
   Targets (y): (2692, 9)


In [20]:
print(f"\n🔍 Checking for NaN values:")
print(f"   NaN in features: {X.isna().sum().sum()}")
print(f"   NaN in targets: {y.isna().sum().sum()}")

print(f"\n📊 Flavor distribution (percentage of recipes with each flavor):")
for flavor in flavor_categories:
    percentage = (y[flavor].sum() / len(y)) * 100
    print(f"   {flavor:<12}: {percentage:6.2f}% ({y[flavor].sum():>4} recipes)")


🔍 Checking for NaN values:
   NaN in features: 0
   NaN in targets: 0

📊 Flavor distribution (percentage of recipes with each flavor):
   bittersweet :  18.05% ( 486 recipes)
   citrus      :  35.55% ( 957 recipes)
   creamy      :   4.38% ( 118 recipes)
   floral      :   4.49% ( 121 recipes)
   fruity      :  27.60% ( 743 recipes)
   herbal      :  16.46% ( 443 recipes)
   savoury     :   3.42% (  92 recipes)
   spicy       :   5.61% ( 151 recipes)
   sweet       :   5.50% ( 148 recipes)


In [21]:
# For multi-label data, we need to ensure stratified split
# We'll create a combined stratification variable
# Create a string representation of all flavor combinations for stratification
y['flavor_combo'] = y[flavor_categories].astype(str).agg(''.join, axis=1)

# Perform stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y[flavor_categories],  # Use only flavor columns for y
    test_size=0.2,
    random_state=42
)

print(f"✅ Data split completed:")
print(f"   Training set: {X_train.shape[0]} recipes")
print(f"   Test set:     {X_test.shape[0]} recipes")
print(f"   Total:        {X_train.shape[0] + X_test.shape[0]} recipes")

# Remove the temporary column
y = y[flavor_categories]

print(f"\n📐 Final dataset shapes:")
print(f"   X_train: {X_train.shape}")
print(f"   X_test:  {X_test.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   y_test:  {y_test.shape}")

✅ Data split completed:
   Training set: 2153 recipes
   Test set:     539 recipes
   Total:        2692 recipes

📐 Final dataset shapes:
   X_train: (2153, 242)
   X_test:  (539, 242)
   y_train: (2153, 9)
   y_test:  (539, 9)


In [24]:
# Dictionary to store all models and results
models = {}
results = {}

optimal_thresholds = {
    'bittersweet': 0.50,
    'citrus': 0.50,      # More aggressive!
    'creamy': 0.60,
    'floral': 0.90,      # Be very strict!
    'fruity': 0.60,
    'herbal': 0.55,
    'savoury': 0.80,
    'spicy': 0.75,
    'sweet': 0.80        # Be strict!
}

# Train separate logistic regression model for each flavor
print("🚀 Training Logistic Regression models (one per flavor)...")
for i, flavor in enumerate(flavor_categories, 1):
    print(f"\n   {i:2}. Training model for: {flavor:<12}")
    
    # Create and train model with L1 regularization for feature selection
    lr_model = LogisticRegression(
        penalty='l1',  # L1 regularization for sparse solutions
        solver='liblinear',  # Works with L1
        max_iter=1000,
        random_state=42,
        class_weight='balanced'  # Handle class imbalance
    )
    
    # Train the model
    lr_model.fit(X_train, y_train[flavor])
    
    # Get probability predictions on validation set (use train for threshold tuning)
    # We'll use cross-validation within train set to find optimal threshold
    from sklearn.model_selection import StratifiedKFold
    
    # Create a validation split from training data
    X_train_sub, X_val, y_train_sub, y_val = train_test_split(
        X_train, y_train[flavor], test_size=0.2, random_state=42, stratify=y_train[flavor]
    )
   
    lr_model.fit(X_train, y_train[flavor])
    
    # Make predictions on TEST set using optimal threshold
    y_test_proba = lr_model.predict_proba(X_test)[:, 1]
    y_pred_optimized = (y_test_proba >= optimal_thresholds[flavor]).astype(int)
    
    # Also get default threshold (0.5) predictions for comparison
    y_pred_default = lr_model.predict(X_test)
    
    # Store model
    models[f'lr_{flavor}'] = {
        'model': lr_model,
        'type': 'logistic_regression',
        'optimal_threshold': optimal_thresholds[flavor]
    }
    
    # Store feature importance (coefficients)
    feature_importance = pd.DataFrame({
        'ingredient': ingredient_columns,
        'coefficient': lr_model.coef_[0]
    }).sort_values('coefficient', key=abs, ascending=False)
    
    # Calculate metrics with OPTIMIZED threshold
    metrics = {
        'accuracy': accuracy_score(y_test[flavor], y_pred_optimized),
        'precision': precision_score(y_test[flavor], y_pred_optimized, zero_division=0),
        'recall': recall_score(y_test[flavor], y_pred_optimized, zero_division=0),
        'f1': f1_score(y_test[flavor], y_pred_optimized, zero_division=0),
        'auc': roc_auc_score(y_test[flavor], y_test_proba)
    }
    
    # Store results
    results[f'lr_{flavor}'] = {
        'threshold': optimal_thresholds[flavor],
        'metrics': metrics,
        'top_ingredients': feature_importance.head(5),
        'model': lr_model
    }
    
    # Print comparison
    print(f"      📊 Performance comparison:")
    print(f"        Threshold: ({optimal_thresholds[flavor]:.2f})")
    print(f"        AUC:       {metrics['auc']:.3f}")
    print(f"        Precision: {metrics['precision']:.3f}")
    print(f"        Recall:    {metrics['recall']:.3f}")
    print(f"        F1-Score:  {metrics['f1']:.3f}")
print("\n✅ All Logistic Regression models trained successfully!")

🚀 Training Logistic Regression models (one per flavor)...

    1. Training model for: bittersweet 
      📊 Performance comparison:
        Threshold: (0.50)
        AUC:       0.960
        Precision: 0.802
        Recall:    0.853
        F1-Score:  0.827

    2. Training model for: citrus      
      📊 Performance comparison:
        Threshold: (0.50)
        AUC:       0.849
        Precision: 0.660
        Recall:    0.865
        F1-Score:  0.749

    3. Training model for: creamy      
      📊 Performance comparison:
        Threshold: (0.60)
        AUC:       0.998
        Precision: 0.917
        Recall:    1.000
        F1-Score:  0.957

    4. Training model for: floral      
      📊 Performance comparison:
        Threshold: (0.90)
        AUC:       0.971
        Precision: 0.458
        Recall:    0.647
        F1-Score:  0.537

    5. Training model for: fruity      
      📊 Performance comparison:
        Threshold: (0.60)
        AUC:       0.863
        Precision: 0.7

In [25]:
print("🚀 Training XGBoost models (one per flavor)...")
for i, flavor in enumerate(flavor_categories, 1):
    print(f"   {i:2}. Training model for: {flavor:<12}", end="")
    
    # Create and train XGBoost model
    xgb_model = XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss',
        scale_pos_weight=len(y_train[y_train[flavor]==0]) / len(y_train[y_train[flavor]==1]) if y_train[flavor].sum() > 0 else 1
    )
    
    # Train the model
    xgb_model.fit(X_train, y_train[flavor])
    
    # Make predictions
    y_pred = xgb_model.predict(X_test)
    y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]
    
    # Store model
    models[f'xgb_{flavor}'] = {
        'model': xgb_model,
        'type': 'xgboost'
    }
    
    # Store feature importance
    feature_importance = pd.DataFrame({
        'ingredient': ingredient_columns,
        'importance': xgb_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Calculate metrics
    results[f'xgb_{flavor}'] = {
        'accuracy': accuracy_score(y_test[flavor], y_pred),
        'precision': precision_score(y_test[flavor], y_pred, zero_division=0),
        'recall': recall_score(y_test[flavor], y_pred, zero_division=0),
        'f1': f1_score(y_test[flavor], y_pred, zero_division=0),
        'auc': roc_auc_score(y_test[flavor], y_pred_proba),
        'top_ingredients': feature_importance.head(5),
        'model': xgb_model
    }
    
    print(f" ✓ (AUC: {results[f'xgb_{flavor}']['auc']:.3f})")
    print(f" ✓ (Precision: {results[f'xgb_{flavor}']['precision']:.3f})")
    print(f" ✓ (Recall: {results[f'xgb_{flavor}']['recall']:.3f})")
    print(f" ✓ (F1-Score: {results[f'xgb_{flavor}']['f1']:.3f})")
    print("============================================================")

print("✅ All XGBoost models trained successfully!")

🚀 Training XGBoost models (one per flavor)...
    1. Training model for: bittersweet  ✓ (AUC: 0.952)
 ✓ (Precision: 0.810)
 ✓ (Recall: 0.853)
 ✓ (F1-Score: 0.831)
    2. Training model for: citrus       ✓ (AUC: 0.881)
 ✓ (Precision: 0.663)
 ✓ (Recall: 0.865)
 ✓ (F1-Score: 0.751)
    3. Training model for: creamy       ✓ (AUC: 0.999)
 ✓ (Precision: 0.815)
 ✓ (Recall: 1.000)
 ✓ (F1-Score: 0.898)
    4. Training model for: floral       ✓ (AUC: 0.921)
 ✓ (Precision: 0.378)
 ✓ (Recall: 0.824)
 ✓ (F1-Score: 0.519)
    5. Training model for: fruity       ✓ (AUC: 0.883)
 ✓ (Precision: 0.736)
 ✓ (Recall: 0.630)
 ✓ (F1-Score: 0.679)
    6. Training model for: herbal       ✓ (AUC: 0.844)
 ✓ (Precision: 0.523)
 ✓ (Recall: 0.659)
 ✓ (F1-Score: 0.583)
    7. Training model for: savoury      ✓ (AUC: 0.839)
 ✓ (Precision: 0.318)
 ✓ (Recall: 0.667)
 ✓ (F1-Score: 0.431)
    8. Training model for: spicy        ✓ (AUC: 0.920)
 ✓ (Precision: 0.615)
 ✓ (Recall: 0.800)
 ✓ (F1-Score: 0.696)
    9. Training mo

Flavor        | LR F1  | XGB F1 | WINNER        | DIFFERENCE
-------------|--------|--------|---------------|-----------
bittersweet  | 0.827  | 0.831  | XGB 🏆        | +0.004 (tie)
citrus       | 0.749  | 0.751  | XGB 🏆        | +0.002 (tie)
creamy       | 0.957  | 0.898  | LR 🏆         | -0.059 (LR better!)
floral       | 0.537  | 0.519  | LR 🏆         | -0.018 (LR better!)
fruity       | 0.664  | 0.679  | XGB 🏆        | +0.015
herbal       | 0.597  | 0.583  | LR 🏆         | -0.014
savoury      | 0.409  | 0.431  | XGB 🏆        | +0.022
spicy        | 0.585  | 0.696  | XGB 🏆🏆      | +0.111 (BIG win!)
sweet        | 0.731  | 0.689  | LR 🏆         | -0.042


Although XgBoost is much better for spicy and a little better for a few other tags, but for now since we need an MVP, we will use all LR models.

In [27]:
print(results['lr_bittersweet'])

{'threshold': 0.5, 'metrics': {'accuracy': 0.9369202226345084, 'precision': 0.801980198019802, 'recall': 0.8526315789473684, 'f1': 0.826530612244898, 'auc': 0.9601825509720247}, 'top_ingredients':                    ingredient  coefficient
188    red_bitter_liqueur_pct    28.001377
99        gentian_liqueur_pct    16.897246
157  orange-red_aperitivo_pct    14.009450
90         fernet_liqueur_pct    13.441718
12   amaro_(e.g._meletti)_pct    12.717095, 'model': LogisticRegression(class_weight='balanced', max_iter=1000, penalty='l1',
                   random_state=42, solver='liblinear')}


In [29]:
import pickle
import json

# Save selected models
models_to_save = {}
for flavor in flavor_categories:
    models_to_save[flavor] = {
        'model': results[f'lr_{flavor}']['model']
    }

# Save everything needed for production
save_data = {
    'models': models_to_save,
    'ingredient_columns': ingredient_columns,  # CRITICAL: Preserve order!
    'flavor_categories': flavor_categories,
    'feature_importance': {f: results[f'lr_{f}']['top_ingredients'].to_dict() 
                          for f in flavor_categories}
}

# Save using pickle
with open('cocktail_flavor_predictor.pkl', 'wb') as f:
    pickle.dump(save_data, f)

# Also save ingredient order separately (for verification)
with open('ingredient_order.json', 'w') as f:
    json.dump(ingredient_columns, f)

print("✅ Models saved successfully!")
print(f"   File: cocktail_flavor_predictor.pkl")
print(f"   Ingredient order saved to: ingredient_order.json")
print(f"   Number of ingredients: {len(ingredient_columns)}")

✅ Models saved successfully!
   File: cocktail_flavor_predictor.pkl
   Ingredient order saved to: ingredient_order.json
   Number of ingredients: 242


In [ ]:
# ============================================================================
# 14. LOAD AND USE MODELS (Example)
# ============================================================================

print("\n" + "=" * 80)
print("14. LOADING AND USING MODELS (Example)")
print("=" * 80)

print("""
# Example of how to load and use the models in production:

import pickle
import numpy as np

# Load the saved models
with open('cocktail_flavor_predictor.pkl', 'rb') as f:
    data = pickle.load(f)

# Get the ingredient order (CRITICAL - must match training order!)
ingredient_columns = data['ingredient_columns']
models = data['models']

def predict_cocktail(ingredient_dict):
    # Convert dictionary to array in correct order
    ingredients = []
    for col in ingredient_columns:
        ingredients.append(ingredient_dict.get(col, 0))
    
    # Make predictions
    predictions = {}
    for flavor, model_info in models.items():
        model = model_info['model']
        prob = model.predict_proba([ingredients])[0][1]
        predictions[flavor] = prob
    
    return predictions

# Example usage
cocktail_ingredients = {'gin_pct': 0.4, 'vermouth_pct': 0.3, 'campari_pct': 0.3}
result = predict_cocktail(cocktail_ingredients)
print(result)
""")